# 10.14 — Latent diffusion & text-to-image
Latent diffusion performs denoising in a compressed representation, then decodes the result into pixels. Autoencoders, VAEs, DDPMs, and classifier-free guidance combine here: text supplies conditioning, diffusion acts in latent space, and the decoder returns pixels. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

SEED = 1014
rng = np.random.default_rng(SEED)


def normalize01(x):
    arr = np.asarray(x, dtype=float)
    lo = float(np.min(arr))
    hi = float(np.max(arr))
    span = hi - lo
    if span < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / span


def synthetic_faces(n=64, size=16, seed=SEED):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    faces = []
    for i in range(n):
        face = 0.25 + 0.25 * np.exp(-2.0 * (xx ** 2 + yy ** 2))
        eye_y = -0.30 + local_rng.normal(0.0, 0.03)
        eye_dx = 0.35 + local_rng.normal(0.0, 0.02)
        mouth_y = 0.45 + local_rng.normal(0.0, 0.04)
        face -= 0.35 * np.exp(-90.0 * ((xx - eye_dx) ** 2 + (yy - eye_y) ** 2))
        face -= 0.35 * np.exp(-90.0 * ((xx + eye_dx) ** 2 + (yy - eye_y) ** 2))
        face += 0.14 * np.exp(-70.0 * (xx ** 2 + (yy - 0.06) ** 2))
        face -= 0.20 * np.exp(-55.0 * (xx ** 2 + (yy - mouth_y) ** 2))
        face += local_rng.normal(0.0, 0.025, size=(size, size))
        faces.append(normalize01(face))
    return np.asarray(faces).reshape(n, size * size)


def load_tiny_faces():
    try:
        from sklearn.datasets import fetch_olivetti_faces
        faces = fetch_olivetti_faces(download_if_missing=False, shuffle=True, random_state=SEED)
        data = faces.data[:64]
        return data, (64, 64), "Olivetti faces cache"
    except Exception:
        return synthetic_faces(), (16, 16), "synthetic face fallback"


def make_f9_ladder(seed=SEED):
    local_rng = np.random.default_rng(seed)
    d1 = local_rng.normal(0.0, 1.0, size=(80, 1))
    d2, _ = make_moons(n_samples=120, noise=0.08, random_state=seed)
    d3, _ = make_blobs(n_samples=144, centers=4, n_features=6, cluster_std=0.65, random_state=seed)
    digits = load_digits()
    d4 = digits.data[:80] / 16.0
    faces, face_shape, face_source = load_tiny_faces()
    ladder = [
        {"name": "D1 gaussian", "data": d1, "shape": (1, 1), "kind": "vector"},
        {"name": "D2 two moons", "data": d2, "shape": (1, 2), "kind": "points"},
        {"name": "D3 mixture", "data": d3, "shape": (2, 3), "kind": "vector"},
        {"name": "D4 sklearn digits", "data": d4, "shape": (8, 8), "kind": "image"},
        {"name": "D5 faces (" + face_source + ")", "data": faces, "shape": face_shape, "kind": "image"},
    ]
    return ladder


def centered_scaled(data):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(np.asarray(data, dtype=float))
    return scaled, scaler


def choose_components(data, cap=8):
    n_samples, n_features = data.shape
    return max(1, min(cap, n_samples - 1, n_features))


def pca_reconstruct(data, n_components):
    n_components = max(1, min(n_components, data.shape[0] - 1, data.shape[1]))
    pca = PCA(n_components=n_components, random_state=SEED)
    z = pca.fit_transform(data)
    recon = pca.inverse_transform(z)
    return recon, z, pca


def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def two_sample_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b), 80)
    a = a[:n]
    b = b[:n]
    aa = pairwise_distances(a, a).mean()
    bb = pairwise_distances(b, b).mean()
    ab = pairwise_distances(a, b).mean()
    return float(2.0 * ab - aa - bb)


def preview_array(ax, sample, shape, title):
    arr = np.asarray(sample)
    if int(np.prod(shape)) == arr.size and shape[0] > 1 and shape[1] > 1:
        ax.imshow(arr.reshape(shape), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.plot(arr.ravel(), marker="o")
    ax.set_title(title, fontsize=9)


def show_ladder_preview(ladder):
    fig, axes = plt.subplots(1, len(ladder), figsize=(14, 2.8))
    for ax, rung in zip(axes, ladder):
        preview_array(ax, rung["data"][0], rung["shape"], rung["name"])
    plt.tight_layout()
    return fig


## The concept, built once (D1)
A latent generator first encodes $x\mapsto z$, denoises $z_T	o z_0$, and decodes $\hat{x}=D(z_0)$. For the lesson numbers, $z=(1,-1)$ and $D=egin{bmatrix}1&0.5\0.2&1\end{bmatrix}$ should give $\hat{x}=(0.5000,-0.8000)$ and $\lVert zVert_2=1.414$.

In [ ]:

def latent_autoencoder_plus_denoise(z, decoder_matrix, noise_scale=0.0, shrink=1.0):
    z = np.asarray(z, dtype=float)
    decoder_matrix = np.asarray(decoder_matrix, dtype=float)
    noisy = z + noise_scale * np.zeros_like(z)
    denoised = shrink * noisy
    decoded = decoder_matrix @ denoised
    latent_norm = float(np.linalg.norm(z))
    return decoded, latent_norm, denoised

z = np.array([1.0, -1.0])
decoder = np.array([[1.0, 0.5], [0.2, 1.0]])
x_hat, latent_norm, z0 = latent_autoencoder_plus_denoise(z, decoder)
assert np.allclose(x_hat, np.array([0.5, -0.8]))
assert round(latent_norm, 3) == 1.414
print("decoded", np.round(x_hat, 4), "latent_norm", round(latent_norm, 3))


The same reusable method can be implemented with PCA as a tiny CPU-only autoencoder: encode to a latent, denoise the latent toward its learned scale, then decode. The reconstruction/denoising error measures how much the compressed path damaged the data.

In [ ]:

def latent_diffusion_simulation(data, latent_dim=None, noise=0.18, shrink=0.92, seed=SEED):
    local_rng = np.random.default_rng(seed)
    if latent_dim is None:
        latent_dim = choose_components(data, cap=8)
    pca = PCA(n_components=latent_dim, random_state=seed)
    z = pca.fit_transform(data)
    z_noisy = z + local_rng.normal(0.0, noise, size=z.shape)
    z_denoised = shrink * z_noisy + (1.0 - shrink) * z.mean(axis=0, keepdims=True)
    generated = pca.inverse_transform(z_denoised)
    metric = mse(data, generated)
    return generated, metric, z, pca


## The dataset ladder (D1–D5)
We use the same F9 ladder inline for every topic: a one-dimensional Gaussian, two moons, a mixture, tiny sklearn digits, and a face rung. The face rung uses a local Olivetti cache when present and otherwise a deterministic no-download synthetic face fallback.

In [ ]:

ladder = make_f9_ladder()
for index, rung in enumerate(ladder, start=1):
    data = rung["data"]
    print(f"D{index}: {rung['name']} | shape={data.shape} | sample_shape={rung['shape']} | kind={rung['kind']}")
show_ladder_preview(ladder)


## Run the SAME method across D1-D5
The same reusable method is applied to every rung and one plan metric is collected per rung.

In [ ]:

results = []
for level, rung in enumerate(ladder, start=1):
    data = rung["data"]
    generated, metric, z, pca = latent_diffusion_simulation(data)
    results.append({"level": level, "name": rung["name"], "metric": metric, "sample": generated[0], "shape": rung["shape"]})
    print(f"D{level} {rung['name']}: reconstruction/denoising error={metric:.5f}, latent_dim={z.shape[1]}")


## Results visualization
The closing figure has generated-sample panels for D1-D5 plus the requested metric curve.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(14, 2.8))
for ax, result in zip(axes, results):
    preview_array(ax, result["sample"], result["shape"], result["name"])
plt.suptitle("Generated or reconstructed samples by rung")
plt.tight_layout()

plt.figure(figsize=(6, 3.2))
plt.plot([r["level"] for r in results], [r["metric"] for r in results], marker="o")
plt.xticks([r["level"] for r in results], [r["name"].split()[0] for r in results])
plt.ylabel("reconstruction / denoising error")
plt.xlabel("F9 rung")
plt.title("Latent diffusion simulation: metric vs. complexity")
plt.grid(True, alpha=0.3)
plt.tight_layout()


## Pitfall on D5: weak autoencoder or changed latent scale
The lesson warns that compression artifacts become the ceiling for generation and that changing latent scale without retraining breaks the schedule. We reproduce the issue by decoding D5 latents after an exaggerated scale change, then fix it by calibrating the scale and using more reconstruction capacity.

In [ ]:

d5 = ladder[-1]
data = d5["data"]
weak_recon, weak_z, weak_pca = pca_reconstruct(data, n_components=2)
wrong_scaled = weak_pca.inverse_transform(3.0 * weak_z)
wrong_error = mse(data, wrong_scaled)
strong_recon, strong_z, strong_pca = pca_reconstruct(data, n_components=choose_components(data, cap=16))
calibrated = strong_pca.inverse_transform(strong_z)
fixed_error = mse(data, calibrated)
print(f"wrong latent-scale error={wrong_error:.5f}")
print(f"fixed calibrated error={fixed_error:.5f}")
fig, axes = plt.subplots(1, 3, figsize=(7, 2.4))
preview_array(axes[0], data[0], d5["shape"], "target D5")
preview_array(axes[1], wrong_scaled[0], d5["shape"], "wrong scale")
preview_array(axes[2], calibrated[0], d5["shape"], "calibrated")
plt.tight_layout()


## Evaluate it + Practice
- Track `reconstruction/denoising error` against a no-skill baseline such as random samples, unconditioned reconstruction, or independent frames.
- Run a cheap sanity check: dimensions match, finite metrics, and generated samples stay in the training range.
- Ablation: change the latent scale or lower the PCA capacity; the metric should get worse.
- Failure signals: D5 looks plausible in one panel but the metric curve jumps, indicating artifacts hidden by small previews.

Practice 1: change the latent or conditioning dimension and predict which rung changes most.


Practice 2: replace the shrink factor with a stronger denoiser and inspect under-smoothing versus over-smoothing.

Practice 3: add a simple text-conditioning vector to the latent mean and measure the reconstruction tradeoff.